In [ ]:
import pandas as pd
import numpy as np
from loguru import logger
from workspace_dani.data_preprocessing.utils.data_handling import load_parquet_table, save_parquet_table
from pathlib import Path

import matplotlib.pyplot as plt

In [ ]:

notebook_path = Path.cwd()
project_root = notebook_path.parents[2]

In [ ]:

lab_path: Path = project_root / Path(
    "intermediary_data/cohorted/sepsis-type-a-cohort/2024-06-23/lab.parquet")

In [ ]:

df_lab = load_parquet_table(parquet_path=lab_path, verbose=False)#.head(n=5000)
df_lab.head()


lab_name_unit_correspondences = df_lab.groupby(by="labname").apply(
    lambda x: x["labmeasurenamesystem"].unique().tolist())

lab_name_unit_correspondences

-> Every lab type has only a distinct measurement interface, i.e. unit

from sklearn.preprocessing import OneHotEncoder


ohe = OneHotEncoder(handle_unknown="error", feature_name_combiner="concat", sparse_output=False)

ohe.fit_transform(df_lab[["labname"]])

lab_type_ohe = pd.DataFrame(ohe.fit_transform(df_lab[["labname"]]))
lab_type_ohe.columns = ohe.get_feature_names_out()

ohe.categories_

lab_type_ohe.head()

lab_types_inversed = ohe.inverse_transform(lab_type_ohe)

OHE works as expected on the lab names (types)

df_lab.head()

In [ ]:
df_lab_dummies = pd.get_dummies(df_lab[["patientunitstayid", "time_offset", "labname", "labresult"]])

# df_lab.groupby(by=["patientunitstayid","time_offset"]).apply(lambda x: print(f"{x.time_offset},{x.labname.tolist()}, {x.labresult.tolist()}"))

df_lab_consolidated = pd.DataFrame()
df_lab_consolidated["lab_types"] = df_lab.groupby(by=["patientunitstayid", "time_offset"]).apply(
    lambda x: x.labname.tolist())
df_lab_consolidated["lab_values"] = df_lab.groupby(by=["patientunitstayid", "time_offset"]).apply(
    lambda x: x.labresult.tolist())

# this df has a multiindex
df_lab_consolidated.head()

Removing the multi index

df_lab_consolidated.reset_index()

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()


mlb.fit_transform(df_lab_consolidated["lab_types"])

### Merging idential time stamps together, by pivoting the dataframe

In [ ]:
df_lab_dummies.head()

In [ ]:
df_lab_dummies.columns

In [ ]:
non_variable_columns = ["patientunitstayid", "time_offset","labresult"]
variable_columns = df_lab_dummies.columns.difference(non_variable_columns)
len(variable_columns)

In [ ]:
#df_lab_dummies["Variable"] = ['Variable_' + str(i) for i in range(1, len(df_lab_dummies) + 1)]

In [ ]:
dict_trans = {}
list_trans = []

dummies_index_unique = df_lab_dummies.drop_duplicates(subset=["patientunitstayid","time_offset"])[["patientunitstayid","time_offset"]]


df_dummies_transf = pd.DataFrame(columns=df_lab_dummies.columns)

In [ ]:
df_dummies_transf_ind = pd.concat([dummies_index_unique, df_dummies_transf]).reset_index(drop=True)

In [ ]:
df_dummies_transf_ind.set_index(["patientunitstayid", "time_offset"], inplace=True)

In [ ]:
df_dummies_transf_ind.index


In [ ]:
from tqdm.auto import tqdm 

for index, row in tqdm(df_lab_dummies.iterrows(), total=len(df_lab_dummies)):
    id = row["patientunitstayid"]
    time_stamp = row["time_offset"]
    # true_col = row[row == True].index.tolist()
    true_col = row[row == True].index.tolist()[0]
    #true_col = "labname_-bands"
    measurement = row["labresult"]
    
    df_dummies_transf_ind.loc[(id, time_stamp), true_col] = measurement
     


In [ ]:

df_lab_proper_time_series = df_dummies_transf_ind.drop(columns="time_offset")



In [ ]:

df_lab_proper_time_series = df_lab_proper_time_series.reset_index()



df_lab_proper_time_series = df_lab_proper_time_series.drop(columns=["labresult"])

In [ ]:
df_lab_proper_time_series.info()

In [ ]:
save_parquet_table(df_in=df_lab_proper_time_series, filename="sepsis_cohort_a_lab_ts", save_path=Path.cwd())